In [1]:
from copy import deepcopy # Actually not needed
from queue import Queue

import sys
sys.setrecursionlimit(10**5)

## Shared functions

In [2]:
def read_input_comma_separated(file_name):
    f = []
    with open(file_name, 'r') as file:
        for row in file:
            f.append(list(row.strip().split(',')))
    return f

def read_input_maze_in_list(file_name):
    f = []
    with open(file_name, 'r') as file:
        for row in file:
            f.append(list(row.strip()))
    return f

def make_list_from_string_input (p_input):
    l = []
    for r in p_input.split('\n'):
        l.append(list(r))

    return l

def draw_field(field):
    for r in range(len(field)):
        row = field[r]
        print(f'row {r}' + ''.join(row), end='\n') 

def draw_field_dict(field_dict):
    key_list = []
    for k in field_dict:
        key_list.append(k)
        key_list.sort()
    
    for k in key_list:
        if k % 1000 == 0:
            print(f'\nrow {k // 1000}: ', end='')
        print(field_dict[k]['v'], end='') 

def encode_coordinates(p_coordinates):
    return int(p_coordinates[0]) * 1000 + int(p_coordinates[1])

def decode_coordinates(p_code):
    return (p_code // 1000, p_code % 1000)

def find_se_maze_list(field):
    for r in range(len(field)):
        for c in range(len(field[0])):
            
            if field[r][c] == 'S':
                s = (r, c)
            
            if field[r][c] == 'E':
                e = (r, c)
                
    return (s, e)

def find_se_maze_dict(p_field_dict):
    
    for k in p_field_dict:
            
            if p_field_dict[k]['v'] == 'S':
                s = k
            
            if p_field_dict[k]['v'] == 'E':
                e = k
                
    return (s, e)

def make_dict_from_maze_list(p_maze_list):
    d_maze = dict()
    for r in range(len(p_maze_list)):
        for c in range(len(p_maze_list[0])):
            d_maze[encode_coordinates((r, c))] = {
                'r': r,
                'c': c,
                'v': p_maze_list[r][c]}
    return d_maze

def get_end_coordinate (p_field_dict):
    for k in p_field_dict.keys():
        if p_field_dict[k]['v'] == 'E':
            return k

def get_nearest_codes(p_field_dict, p_code):
    output = []
    (x, y) = decode_coordinates(p_code)
    for (xd, yd) in ((0, -1), (0, 1), (-1, 0), (1, 0)):
        c = encode_coordinates((x + xd, y + yd))
        if c in p_field_dict and p_field_dict[c]['v'] != '#':
            output.append(c)
        else:
            output.append(-1)
    return output    

def clear_dict_score_till_end(p_field_dict):
    for k in p_field_dict.keys():
        if 'score_till_end' in p_field_dict[k]:
            del p_field_dict[k]['score_till_end']

def clear_dict_cheats(p_field_dict):
    for k in p_field_dict.keys():
        if 'cheats' in p_field_dict[k]:
            del p_field_dict[k]['cheats']

def dbg(debug, p_string):
    if debug:
        print (p_string)    

def show_dict_non_wall_positions(p_field_dict):
    for k in p_field_dict:
        if field_dict[k]['v'] != '#':
            print(f'{k}: {p_field_dict[k]}\n')

## Specific code

In [3]:
def get_nearest_cheats(p_field_dict, p_code, debug=False):
    output = dict()
    infinity = calc_infinity_for_maze_dict(p_field_dict)
    
    # If there is no path backwards to the start - no cheats
    if p_field_dict[p_code]['score_till_start'] > infinity:
        return output
        
    (x, y) = decode_coordinates(p_code)
    directions = {'left': (0, -1), 'right': (0, 1), 'up': (-1, 0), 'down': (1, 0)}
    infinity = calc_infinity_for_maze_dict(p_field_dict)
    for key in directions:
        if debug:
            print('\n' + key, end=' ')
        (xd, yd) = directions[key]
        cw = encode_coordinates((x + xd, y + yd))
        c = encode_coordinates((x + xd*2, y + yd*2))
        if debug:
            print(f'c = {c}', end='')
            try:
                print(f", p_field_dict[{c}] = {p_field_dict[c]['v']}", end='')
            except:
                None
            print(f', cw = {cw}', end='')
            try:
                print(f", p_field_dict[{cw}] = {p_field_dict[cw]['v']}", end='')
            except:
                None
        if c in p_field_dict and p_field_dict[cw]['v'] == '#' and p_field_dict[c]['v'] in ('.', 'E') and p_field_dict[c]['score_till_end'] < infinity:
            saving = p_field_dict[p_code]['score_till_end'] - p_field_dict[c]['score_till_end'] - 2
            saving = saving if saving > 0 else 0
            if debug:
                print(f', reduced_score = {saving}', end='')
            output[key] = saving
    return output 
    
def gen_cheats(p_field_dict):
    for k in p_field_dict:
        if p_field_dict[k]['v'] in ('.', 'S'):
            cheat_dict = get_nearest_cheats(field_dict, k)
            p_field_dict[k]['cheats'] = cheat_dict

def count_cheat_cases(p_field_dict, lower_limit):
    infinity = calc_infinity_for_maze_dict(p_field_dict)     
    result = dict()
    directions = {'left': (0, -1), 'right': (0, 1), 'up': (-1, 0), 'down': (1, 0)}
    for k in p_field_dict:
        if p_field_dict[k]['v'] in ('.', 'S'): # and p_field_dict[k]['score_till_end'] < infinity:
            for dir_k in directions:
                if dir_k in p_field_dict[k]['cheats'] and p_field_dict[k]['cheats'][dir_k] >= lower_limit:
                    saving = p_field_dict[k]['cheats'][dir_k]
                    if saving in result:
                        result[saving] += 1
                    else:
                        result[saving] = 1
    return result

def count_cheats(p_field_dict, lower_limit):
    infinity = calc_infinity_for_maze_dict(p_field_dict) 
    counter = 0
    directions = {'left': (0, -1), 'right': (0, 1), 'up': (-1, 0), 'down': (1, 0)}
    for k in p_field_dict:
        if p_field_dict[k]['v'] in ('.', 'S'): # and p_field_dict[k]['score_till_end'] < infinity:
            for dir_k in directions:
                if dir_k in p_field_dict[k]['cheats']:
                    saving = p_field_dict[k]['cheats'][dir_k]
                    if saving >= lower_limit:
                        #print (f"p_field_dict[{k}]['cheats'][{dir_k}]: {p_field_dict[k]['cheats'][dir_k]}")
                        counter += 1
    return counter

def calc_infinity_for_maze_dict(p_field_dict):
    max_code = max(field_dict)
    max_x = max_code // 1000
    max_y = max_code % 1000
    return max_x * max_y + 1_000_000_000

def calc_distance_to_end (p_field_dict, p_code, p_previous_codes, debug=False):

    (left_code, right_code, up_code, down_code) = get_nearest_codes(p_field_dict, p_code)
    infinity = calc_infinity_for_maze_dict(p_field_dict)
    
    dbg(debug, f'Start calculation of p_code = {p_code} with (left_code, right_code, up_code, down_code) = ({left_code}, {right_code}, {up_code}, {down_code})')

    # If it is the end position and it is still not calculated - put value of 0 in it
    if not 'score_till_end' in p_field_dict[p_code] and p_field_dict[p_code]['v'] == 'E':
        p_field_dict[p_code]['score_till_end'] = 0
        return
    
    for next_code in (left_code, right_code, up_code, down_code):

        # If the new_cde is on the field, no cycle and we can go there
        if  next_code >= 0 \
            and next_code not in p_previous_codes  \
            and p_field_dict[next_code]['v'] in ('.', 'S', 'E'): 

            if 'score_till_end' not in p_field_dict[next_code]: 
                calc_distance_to_end (p_field_dict, next_code, p_previous_codes + [p_code], debug)  

            dbg (debug, f"next_code: p_field_dict[{next_code}][{'score_till_end'}]= {p_field_dict[next_code]['score_till_end']}")     

            # If no score is calculated yet for p_code or the current p_code is higher than the one comming from the next_code
            # set the score

            if 'score_till_end' in p_field_dict[next_code]:
            
                if 'score_till_end' not in p_field_dict[p_code] or \
                    p_field_dict[p_code]['score_till_end'] > p_field_dict[next_code]['score_till_end'] + 1:
                    
                        p_field_dict[p_code]['score_till_end'] = p_field_dict[next_code]['score_till_end'] + 1

    if 'score_till_end' not in p_field_dict[p_code] :
        dbg(debug, f'\nDuring calculation of p_code = {p_code}: for next_code = {next_code} there is no calculated score_till_end.')
        p_field_dict[p_code]['score_till_end'] = infinity
        
        #print(f'p_previous_codes at that moment is  {p_previous_codes}\n')
            
    #dbg (debug, f"p_field_dict[{p_code}][{'score_till_end'}]= {p_field_dict[p_code]['score_till_end']}")

def calc_distance_to_end_all_keys (p_field_dict, debug=False):
    for k in p_field_dict:
        if p_field_dict[k]['v'] in ('.', 'S', 'E'):
            calc_distance_to_end (p_field_dict, k, [], debug)

def calc_distance_to_start (p_field_dict, p_code, p_previous_codes, debug=False):

    (left_code, right_code, up_code, down_code) = get_nearest_codes(p_field_dict, p_code)
    infinity = calc_infinity_for_maze_dict(p_field_dict)
    
    #dbg(debug, f'Start calculation of p_code = {p_code} with (left_code, right_code, up_code, down_code) = ({left_code}, {right_code}, {up_code}, {down_code})')

    # If it is the end position and it is still not calculated - put value of 0 in it
    if not 'score_till_start' in p_field_dict[p_code] and p_field_dict[p_code]['v'] == 'S':
        p_field_dict[p_code]['score_till_start'] = 0
        return
    
    for next_code in (left_code, right_code, up_code, down_code):

        # If the new_cde is on the field, no cycle and we can go there
        if  next_code >= 0 \
            and next_code not in p_previous_codes  \
            and p_field_dict[next_code]['v'] in ('.', 'S', 'E'): 

            if 'score_till_start' not in p_field_dict[next_code]: 
                calc_distance_to_start (p_field_dict, next_code, p_previous_codes + [p_code], debug)  

            #dbg (debug, f"next_code: p_field_dict[{next_code}][{'score_till_end'}]= {p_field_dict[next_code]['score_till_end']}")     

            # If no score is calculated yet for p_code or the current p_code is higher than the one comming from the next_code
            # set the score

            if 'score_till_start' in p_field_dict[next_code]:
            
                if 'score_till_start' not in p_field_dict[p_code] or \
                    p_field_dict[p_code]['score_till_start'] > p_field_dict[next_code]['score_till_start'] + 1:
                    
                        p_field_dict[p_code]['score_till_start'] = p_field_dict[next_code]['score_till_start'] + 1

    if 'score_till_start' not in p_field_dict[p_code] :
        p_field_dict[p_code]['score_till_start'] = infinity
        #print(f'\n\nDuring calculation of p_code = {p_code}: for next_code = {next_code} there is no calculated score_till_end.', end = '\n')
        #print(f'p_previous_codes at that moment is  {p_previous_codes}\n')
            
    #dbg (debug, f"p_field_dict[{p_code}][{'score_till_end'}]= {p_field_dict[p_code]['score_till_end']}")

def calc_distance_to_start_all_keys (p_field_dict, debug=False):
    for k in p_field_dict:
        if p_field_dict[k]['v'] in ('.', 'S', 'E'):
            calc_distance_to_start (p_field_dict, k, [], debug)

In [6]:
input_str = """###############
#...#...#.....#
#.#.#.#.#.###.#
#S#...#.#.#...#
#######.#.#.###
#######.#.#...#
#######.#.###.#
###..E#...#...#
###.#######.###
#...###...#...#
#.#####.#.###.#
#.#...#.#.#...#
#.#.#.#.#.#.###
#...#...#...###
###############"""

#field_dict = make_dict_from_maze_list(make_list_from_string_input(input_str))
field_dict = make_dict_from_maze_list(read_input_maze_in_list('input.txt'))

In [7]:
(v_start, v_end) = find_se_maze_dict(field_dict) #(97067, 117059)
calc_distance_to_end (field_dict, v_start, [], debug=False)
calc_distance_to_start (field_dict, v_end, [], debug=False)
gen_cheats(field_dict)
count_cheats(field_dict, 100)

1296

516 is too low

1199 is too low

1308 is too high

Answer: 1296

In [9]:
def get_distance_nearest_cheats(p_field_dict, p_code, p_dist, debug=False):
    output = dict()
    infinity = calc_infinity_for_maze_dict(p_field_dict)
    
    # If there is no path backwards to the start - no cheats
    if p_field_dict[p_code]['score_till_start'] > infinity:
        return output
        
    (x, y) = decode_coordinates(p_code)
    destinations = dict()
    
    for row_dist in range(-p_dist, p_dist+1):
        for col_dist in range(-p_dist, p_dist+1):
            if abs(row_dist) + abs(col_dist) <= p_dist:
                dest_code = encode_coordinates((x + row_dist, y + col_dist))
                
                if dest_code in p_field_dict and p_field_dict[dest_code]['v'] in ('.', 'E') and p_field_dict[dest_code]['score_till_end'] < infinity:
                    saving = p_field_dict[p_code]['score_till_end'] - p_field_dict[dest_code]['score_till_end'] - (abs(row_dist) + abs(col_dist))
                    saving = saving if saving > 0 else 0
                    if saving >= 50:
                        output[dest_code] = saving

    return output 

In [10]:
def gen_distance_cheats(p_field_dict):
    for k in p_field_dict:
        if p_field_dict[k]['v'] in ('.', 'S'):
            cheat_dict = get_distance_nearest_cheats(field_dict, k, 20)
            p_field_dict[k]['distance_cheats'] = cheat_dict

In [11]:
gen_distance_cheats(field_dict)

In [12]:
def count_distance_cheat_cases(p_field_dict):
    infinity = calc_infinity_for_maze_dict(p_field_dict)     
    result = dict()
    for k in p_field_dict:
        if p_field_dict[k]['v'] in ('.', 'S') and 'distance_cheats' in p_field_dict[k] :
            for code in p_field_dict[k]['distance_cheats']:
                saving = p_field_dict[k]['distance_cheats'][code]
                if saving >= 50:
                    if saving in result:
                        result[saving] += 1
                    else:
                        result[saving] = 1
    return result

In [13]:
d = count_distance_cheat_cases(field_dict)
d_keys = [k for k in d.keys()]
for k in sorted(d_keys):
    if k >= 50:
        print (f'- There are {d[k]} cheats that save {k} picoseconds.')

- There are 8250 cheats that save 50 picoseconds.
- There are 14361 cheats that save 52 picoseconds.
- There are 7871 cheats that save 54 picoseconds.
- There are 13679 cheats that save 56 picoseconds.
- There are 7797 cheats that save 58 picoseconds.
- There are 14226 cheats that save 60 picoseconds.
- There are 7511 cheats that save 62 picoseconds.
- There are 12660 cheats that save 64 picoseconds.
- There are 7243 cheats that save 66 picoseconds.
- There are 12888 cheats that save 68 picoseconds.
- There are 7083 cheats that save 70 picoseconds.
- There are 12391 cheats that save 72 picoseconds.
- There are 6907 cheats that save 74 picoseconds.
- There are 12286 cheats that save 76 picoseconds.
- There are 6716 cheats that save 78 picoseconds.
- There are 12350 cheats that save 80 picoseconds.
- There are 6372 cheats that save 82 picoseconds.
- There are 11304 cheats that save 84 picoseconds.
- There are 6286 cheats that save 86 picoseconds.
- There are 11411 cheats that save 88 pic

In [ ]:
show_dict_non_wall_positions(field_dict)

In [14]:
def count_distant_cheats(p_field_dict, lower_limit):
    counter = 0
    for k in p_field_dict:
        if p_field_dict[k]['v'] in ('.', 'S') and 'distance_cheats' in p_field_dict[k] :
            for code in p_field_dict[k]['distance_cheats']:
                saving = p_field_dict[k]['distance_cheats'][code]
                if saving >= lower_limit:
                    counter += 1
    return counter

In [16]:
count_distant_cheats(field_dict, 100)

977665

1214521 is Too high

Answer: 977665